In [ ]:
# -*- coding: utf-8 -*-
"""
ANÁLISE COMPLETA EM FAIXAS DE 20 kHz — RF PONTO A PONTO vs PARK

Este script faz tudo em uma única execução:

1) Carrega a base .pkl
2) Separa automaticamente as colunas de frequência
3) Analisa a base em faixas de 20 em 20 kHz
4) Aplica dois métodos de compensação térmica:
   - Random Forest ponto a ponto
   - Park
5) Calcula RMSD e CCDM para dano 0, dano 1 e dano 2
6) Salva CSVs com os resultados
7) Gera:
   - histogramas grandes RF vs Park
   - ranking das melhores faixas
   - gráficos de RMSD e CCDM por faixa
   - gráficos RF vs Park por dano
   - heatmaps de RMSD
   - heatmaps de CCDM
   - heatmaps do score combinado

Observação importante:
- O código usa a curva saudável na temperatura REF_TEMP como referência.
- Se não houver dano 0 exatamente na REF_TEMP, ele usa a mediana geral do dano 0.

Autor: adaptado para Luiz Eduardo Abdala José
"""

# ============================================================
# 1) IMPORTS
# ============================================================
import os
import re
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


# ============================================================
# 2) CONFIGURAÇÕES PRINCIPAIS
# ============================================================

# Arquivo da base
ARQ_BASE = "base-completo--.pkl"

# Temperatura de referência
REF_TEMP = 30

# Pasta principal de saída
PASTA_SAIDA = "resultados_rf_park_20khz_heatmaps"
os.makedirs(PASTA_SAIDA, exist_ok=True)

# Subpastas
PASTA_HIST = os.path.join(PASTA_SAIDA, "histogramas_20khz")
PASTA_GRAF = os.path.join(PASTA_SAIDA, "graficos_metricas")
PASTA_HEAT = os.path.join(PASTA_SAIDA, "heatmaps")

for p in [PASTA_HIST, PASTA_GRAF, PASTA_HEAT]:
    os.makedirs(p, exist_ok=True)

# ------------------------------------------------------------
# Faixas de frequência
# ------------------------------------------------------------
# Se USAR_FAIXA_TOTAL_DA_BASE = True, o código detecta automaticamente
# a menor e a maior frequência existentes na base.
# Se quiser forçar uma faixa manual, coloque False e ajuste os valores abaixo.
USAR_FAIXA_TOTAL_DA_BASE = True

# Só são usados se USAR_FAIXA_TOTAL_DA_BASE = False
FREQ_MIN_GLOBAL_KHZ = 30
FREQ_MAX_GLOBAL_KHZ = 100

# Aqui está o principal pedido: faixas de 20 em 20 kHz.
LARGURA_FAIXA_KHZ = 20
PASSO_FAIXA_KHZ = 20

# Se True, também cria uma faixa final menor caso sobre um pedaço no fim.
# Exemplo: 30-100 com largura 20 gera 30-50, 50-70, 70-90 e 90-100.
INCLUIR_FAIXA_FINAL_INCOMPLETA = True

# Número mínimo de pontos de frequência por faixa.
MIN_PONTOS_FREQ = 5

# ------------------------------------------------------------
# Histogramas
# ------------------------------------------------------------
GERAR_HISTOGRAMAS_TODAS_AS_FAIXAS = True
GERAR_HISTOGRAMA_MELHOR_FAIXA = True
N_TEMPS_HIST = 6
SEED_TEMPS = 42

# ------------------------------------------------------------
# Suavização
# ------------------------------------------------------------
SMOOTH_WIN = 5

# ------------------------------------------------------------
# Park
# ------------------------------------------------------------
PARK_MAX_SHIFT_FRAC = 0.25
PARK_SMOOTH_WIN = 5
PARK_NSTEPS = 151

# ------------------------------------------------------------
# Random Forest ponto a ponto
# ------------------------------------------------------------
RF_COMP_POINT_PARAMS = dict(
    n_estimators=250,
    max_depth=10,
    min_samples_leaf=2,
    min_samples_split=4,
    max_features="sqrt",
    n_jobs=-1,
    random_state=0,
)

# ------------------------------------------------------------
# Salvar PDF além de PNG
# ------------------------------------------------------------
SALVAR_PDF = True


# ============================================================
# 3) CONFIGURAÇÃO VISUAL
# ============================================================

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 22,
    "axes.labelsize": 26,
    "axes.titlesize": 28,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 18,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

NOME_METODO = {
    "RF_ponto_a_ponto": "RF ponto a ponto",
    "Park": "Park",
}

CORES_DANO = {
    0: "tab:blue",
    1: "tab:orange",
    2: "tab:red",
}

ESTILO_METODO = {
    "RF_ponto_a_ponto": "-",
    "Park": "--",
}

MARCADORES_METODO = {
    "RF_ponto_a_ponto": "o",
    "Park": "s",
}


# ============================================================
# 4) FUNÇÕES DE FREQUÊNCIA E PRÉ-PROCESSAMENTO
# ============================================================

def extract_freq_hz(col):
    """Extrai a frequência de colunas no formato f_30000Hz."""
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None


def listar_colunas_frequencia(df):
    """Retorna colunas de frequência e frequências em Hz, ordenadas."""
    cols = []
    freqs = []

    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None:
            cols.append(c)
            freqs.append(f)

    if len(cols) == 0:
        raise ValueError("Não encontrei colunas no formato f_XXXXXHz.")

    order = np.argsort(freqs)
    cols = [cols[i] for i in order]
    freqs = np.asarray(freqs, dtype=float)[order]

    return cols, freqs


def get_freq_columns(df, fmin_khz, fmax_khz):
    """Seleciona colunas de frequência dentro da faixa escolhida."""
    cols, freqs = [], []

    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None:
            f_khz = f / 1e3
            if fmin_khz <= f_khz <= fmax_khz:
                cols.append(c)
                freqs.append(f)

    if len(cols) == 0:
        return [], np.array([])

    order = np.argsort(freqs)
    cols = [cols[i] for i in order]
    freqs = np.asarray(freqs, dtype=float)[order]

    return cols, freqs


def definir_limites_frequencia(df):
    """Define automaticamente a menor e a maior frequência disponíveis."""
    _, freqs_hz = listar_colunas_frequencia(df)
    fmin = float(np.min(freqs_hz) / 1e3)
    fmax = float(np.max(freqs_hz) / 1e3)
    return fmin, fmax


def gerar_faixas(fmin_global, fmax_global, largura, passo, incluir_final=True):
    """
    Gera faixas do tipo:
    30-50, 50-70, 70-90...

    Se incluir_final=True, uma sobra final também é criada.
    Exemplo: 30-100 gera 30-50, 50-70, 70-90 e 90-100.
    """
    faixas = []
    ini = float(fmin_global)
    fmax_global = float(fmax_global)

    while ini + largura <= fmax_global + 1e-9:
        faixas.append((float(ini), float(ini + largura)))
        ini += passo

    if incluir_final and ini < fmax_global:
        if len(faixas) == 0 or faixas[-1][1] < fmax_global:
            faixas.append((float(ini), float(fmax_global)))

    return faixas


def moving_average(arr, win):
    """Média móvel simples para suavizar a curva."""
    arr = np.asarray(arr, dtype=float)

    if win <= 1 or win % 2 == 0:
        return arr.copy()

    pad = win // 2
    arr_pad = np.pad(arr, (pad, pad), mode="edge")
    kernel = np.ones(win) / win
    smooth = np.convolve(arr_pad, kernel, mode="valid")

    if len(smooth) > len(arr):
        smooth = smooth[:len(arr)]
    elif len(smooth) < len(arr):
        smooth = np.pad(smooth, (0, len(arr) - len(smooth)), mode="edge")

    return smooth


def add_extra_features_matrix(X):
    """
    Entrada do RF ponto a ponto:
    curva inteira + média + desvio padrão + amplitude.
    """
    X = np.asarray(X, dtype=float)
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True)
    amp = (X.max(axis=1) - X.min(axis=1)).reshape(-1, 1)
    return np.hstack([X, mu, sd, amp])


# ============================================================
# 5) MÉTRICAS
# ============================================================

def rmsd(y, ref):
    """RMSD: quanto menor, mais perto da referência."""
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)
    return float(np.sqrt(np.mean((y - ref) ** 2)))


def ccdm(y, ref):
    """
    CCDM = 1 - correlação de Pearson.
    Quanto menor, mais parecida é a forma da curva.
    """
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)

    y0 = y - np.mean(y)
    r0 = ref - np.mean(ref)

    num = float(np.sum(y0 * r0))
    den = float(np.sqrt(np.sum(y0 ** 2) * np.sum(r0 ** 2))) + 1e-18

    corr = num / den
    return float(1 - corr)


def calcular_metricas(df_comp, fcols, y_ref):
    """Calcula RMSD e CCDM de cada curva compensada contra a referência saudável."""
    X = df_comp[fcols].to_numpy(float)

    df2 = df_comp[["temperatura_c", "falha"]].copy()
    df2["RMSD"] = [rmsd(x, y_ref) for x in X]
    df2["CCDM"] = [ccdm(x, y_ref) for x in X]

    return df2


def resumir_metricas_por_dano(df_metricas, metodo, fmin_khz, fmax_khz, n_freq):
    """Resumo médio por dano para uma faixa e um método."""
    rows = []

    for d in sorted(df_metricas["falha"].unique()):
        sub = df_metricas[df_metricas["falha"] == d]

        rows.append({
            "faixa_min_khz": fmin_khz,
            "faixa_max_khz": fmax_khz,
            "faixa_centro_khz": 0.5 * (fmin_khz + fmax_khz),
            "largura_khz": fmax_khz - fmin_khz,
            "n_pontos_freq": int(n_freq),
            "metodo": metodo,
            "falha": int(d),
            "RMSD_medio": sub["RMSD"].mean(),
            "RMSD_std": sub["RMSD"].std(),
            "CCDM_medio": sub["CCDM"].mean(),
            "CCDM_std": sub["CCDM"].std(),
            "n_amostras": len(sub),
        })

    return rows


# ============================================================
# 6) REFERÊNCIA SAUDÁVEL
# ============================================================

def curva_referencia_saudavel(df, fcols):
    """
    Curva de referência = mediana das curvas sem dano na temperatura REF_TEMP.
    Se não existir REF_TEMP, usa a mediana de todas as curvas sem dano.
    """
    df_sem = df[df["falha"] == 0]

    if len(df_sem) == 0:
        raise ValueError("Não existe nenhuma amostra com falha == 0 para montar a referência saudável.")

    pool = df_sem.loc[np.isclose(df_sem["temperatura_c"], REF_TEMP), fcols].to_numpy(float)

    if len(pool) > 0:
        return np.median(pool, axis=0)

    print(f"⚠️ Não achei dados sem dano em {REF_TEMP}°C. Usando mediana geral sem dano.")
    return np.median(df_sem[fcols].to_numpy(float), axis=0)


# ============================================================
# 7) MÉTODO 1 — RF DIRETO PONTO A PONTO
# ============================================================

def compensar_rf_direto(df, fcols):
    """
    Treina o RF apenas com dados sem dano.

    Entrada:
    - curva medida
    - features simples da curva
    - temperatura

    Saída aprendida:
    - correção necessária para levar a curva sem dano até a referência saudável.
    """
    df_sem = df[df["falha"] == 0]

    if len(df_sem) == 0:
        raise ValueError("RF precisa de amostras sem dano, isto é, falha == 0.")

    X_sem = df_sem[fcols].to_numpy(float)
    T_sem = df_sem["temperatura_c"].to_numpy(float)

    y_ref = curva_referencia_saudavel(df, fcols)

    Y_target = y_ref[None, :] - X_sem

    X_aug = add_extra_features_matrix(X_sem)
    X_in = np.hstack([X_aug, T_sem.reshape(-1, 1)])

    rf = RandomForestRegressor(**RF_COMP_POINT_PARAMS)
    rf.fit(X_in, Y_target)

    X_all = df[fcols].to_numpy(float)
    T_all = df["temperatura_c"].to_numpy(float)

    X_aug_all = add_extra_features_matrix(X_all)
    X_in_all = np.hstack([X_aug_all, T_all.reshape(-1, 1)])

    Y_comp = X_all + rf.predict(X_in_all)

    for i in range(len(Y_comp)):
        Y_comp[i] = moving_average(Y_comp[i], SMOOTH_WIN)

    df2 = df.copy()
    df2[fcols] = Y_comp

    return df2, y_ref


# ============================================================
# 8) MÉTODO 2 — PARK
# ============================================================

def shift_interp(x, f, tau):
    """Deslocamento horizontal por interpolação."""
    f_shift = f + tau
    return np.interp(f, f_shift, x, left=x[0], right=x[-1])


def park_single(x, ref, fHz):
    """
    Park simplificado:
    - testa deslocamentos horizontais
    - aplica deslocamento vertical médio
    - escolhe a combinação com menor erro quadrático contra a referência
    """
    df_band = fHz[-1] - fHz[0]
    tau_max = PARK_MAX_SHIFT_FRAC * df_band

    best_err = np.inf
    best_tau = 0.0
    best_dS = 0.0

    for tau in np.linspace(-tau_max, tau_max, PARK_NSTEPS):
        x_shift = shift_interp(x, fHz, tau)
        dS = np.mean(ref - x_shift)
        err = np.sum((ref - (x_shift + dS)) ** 2)

        if err < best_err:
            best_err = err
            best_tau = tau
            best_dS = dS

    y = shift_interp(x, fHz, best_tau) + best_dS
    y = moving_average(y, PARK_SMOOTH_WIN)

    return y


def compensar_park(df, fcols, fHz):
    """Aplica Park curva a curva."""
    y_ref = curva_referencia_saudavel(df, fcols)

    X_all = df[fcols].to_numpy(float)
    Y = np.zeros_like(X_all)

    for i in range(len(X_all)):
        Y[i] = park_single(X_all[i], y_ref, fHz)

    df2 = df.copy()
    df2[fcols] = Y

    return df2, y_ref


# ============================================================
# 9) EXECUTAR UMA FAIXA
# ============================================================

def executar_uma_faixa(df_base, fmin_khz, fmax_khz):
    """Roda RF e Park para uma faixa específica."""
    fcols, fHz = get_freq_columns(df_base, fmin_khz, fmax_khz)

    if len(fcols) < MIN_PONTOS_FREQ:
        raise ValueError(
            f"Poucos pontos na faixa {fmin_khz:.1f}-{fmax_khz:.1f} kHz: {len(fcols)} pontos."
        )

    df_use = df_base[["temperatura_c", "falha"] + fcols].copy()

    print(f"\n🔹 Rodando faixa {fmin_khz:.1f}-{fmax_khz:.1f} kHz | {len(fcols)} pontos")

    t0 = time.time()

    df_rf_comp, yref_rf = compensar_rf_direto(df_use, fcols)
    df_park_comp, yref_park = compensar_park(df_use, fcols, fHz)

    df_rf_met = calcular_metricas(df_rf_comp, fcols, yref_rf)
    df_park_met = calcular_metricas(df_park_comp, fcols, yref_park)

    dt = time.time() - t0
    print(f"✅ Concluído em {dt:.1f} s")

    resumo = []
    resumo += resumir_metricas_por_dano(df_rf_met, "RF_ponto_a_ponto", fmin_khz, fmax_khz, len(fcols))
    resumo += resumir_metricas_por_dano(df_park_met, "Park", fmin_khz, fmax_khz, len(fcols))

    df_resumo = pd.DataFrame(resumo)

    return df_rf_met, df_park_met, df_resumo


# ============================================================
# 10) CRITÉRIO DE RANKING
# ============================================================

def _zscore_col(s):
    s = pd.Series(s, dtype=float)
    std = s.std()
    if std == 0 or np.isnan(std):
        return s * 0.0
    return (s - s.mean()) / std


def montar_tabela_ranking(df_resumo):
    """
    Monta ranking por método e faixa.

    Ideia do score:
    - RMSD baixo no dano 0 é bom
    - CCDM baixo no dano 0 é bom
    - Separação D1-D0 grande é boa
    - Separação D2-D1 grande é boa
    - Inversão da ordem D0 < D1 < D2 é ruim

    Score menor = melhor faixa.
    """
    rows = []

    for (fmin, fmax, metodo), g in df_resumo.groupby(["faixa_min_khz", "faixa_max_khz", "metodo"]):
        gd = g.set_index("falha")

        if not all(d in gd.index for d in [0, 1, 2]):
            continue

        rmsd0 = gd.loc[0, "RMSD_medio"]
        rmsd1 = gd.loc[1, "RMSD_medio"]
        rmsd2 = gd.loc[2, "RMSD_medio"]

        ccdm0 = gd.loc[0, "CCDM_medio"]
        ccdm1 = gd.loc[1, "CCDM_medio"]
        ccdm2 = gd.loc[2, "CCDM_medio"]

        sep_rmsd_10 = rmsd1 - rmsd0
        sep_rmsd_21 = rmsd2 - rmsd1
        sep_ccdm_10 = ccdm1 - ccdm0
        sep_ccdm_21 = ccdm2 - ccdm1

        inversao_rmsd = int(sep_rmsd_10 <= 0) + int(sep_rmsd_21 <= 0)
        inversao_ccdm = int(sep_ccdm_10 <= 0) + int(sep_ccdm_21 <= 0)

        rows.append({
            "faixa_min_khz": fmin,
            "faixa_max_khz": fmax,
            "faixa_centro_khz": 0.5 * (fmin + fmax),
            "faixa_label": f"{fmin:.0f}-{fmax:.0f} kHz",
            "metodo": metodo,
            "RMSD_D0": rmsd0,
            "RMSD_D1": rmsd1,
            "RMSD_D2": rmsd2,
            "CCDM_D0": ccdm0,
            "CCDM_D1": ccdm1,
            "CCDM_D2": ccdm2,
            "sep_RMSD_D1_D0": sep_rmsd_10,
            "sep_RMSD_D2_D1": sep_rmsd_21,
            "sep_CCDM_D1_D0": sep_ccdm_10,
            "sep_CCDM_D2_D1": sep_ccdm_21,
            "inversoes_RMSD": inversao_rmsd,
            "inversoes_CCDM": inversao_ccdm,
            "inversoes_total": inversao_rmsd + inversao_ccdm,
        })

    rank = pd.DataFrame(rows)

    if len(rank) == 0:
        raise ValueError("Não foi possível montar ranking. Verifique se existem danos 0, 1 e 2.")

    rank["score"] = (
        1.00 * _zscore_col(rank["RMSD_D0"]) +
        1.00 * _zscore_col(rank["CCDM_D0"]) -
        0.50 * _zscore_col(rank["sep_RMSD_D1_D0"]) -
        0.50 * _zscore_col(rank["sep_RMSD_D2_D1"]) -
        0.50 * _zscore_col(rank["sep_CCDM_D1_D0"]) -
        0.50 * _zscore_col(rank["sep_CCDM_D2_D1"]) +
        1.50 * rank["inversoes_total"]
    )

    rank = rank.sort_values("score", ascending=True).reset_index(drop=True)
    return rank


# ============================================================
# 11) FUNÇÕES DE PLOT
# ============================================================

def estilo_eixos(ax):
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=20)


def salvar_fig(fig, pasta, nome_base):
    png_path = os.path.join(pasta, nome_base + ".png")
    fig.savefig(png_path, dpi=600, bbox_inches="tight")

    if SALVAR_PDF:
        pdf_path = os.path.join(pasta, nome_base + ".pdf")
        fig.savefig(pdf_path, bbox_inches="tight")
        print(f"✅ Salvo:\n{png_path}\n{pdf_path}")
    else:
        print(f"✅ Salvo:\n{png_path}")


def erro_valido(y, yerr):
    y = np.asarray(y, dtype=float)
    yerr = np.asarray(yerr, dtype=float)
    yerr = np.where(np.isfinite(yerr), yerr, 0.0)
    return y, yerr


def histograma_rf_park_temperaturas_validas_2painel(df_rf, df_park, fmin_khz, fmax_khz, output_dir):
    """Histograma grande com RMSD e CCDM para uma faixa."""
    required_cols = ["falha", "temperatura_c", "RMSD", "CCDM"]
    for name, data in [("df_rf", df_rf), ("df_park", df_park)]:
        missing = [c for c in required_cols if c not in data.columns]
        if missing:
            raise ValueError(f"{name} está sem as colunas: {missing}")

    danos = [0, 1, 2]

    temps_validas = None
    for data in [df_rf, df_park]:
        sets = []
        for d in danos:
            sets.append(set(data.loc[data["falha"] == d, "temperatura_c"].unique()))
        validas_data = set.intersection(*sets)
        temps_validas = validas_data if temps_validas is None else temps_validas & validas_data

    temps_validas = sorted(list(temps_validas))

    if len(temps_validas) == 0:
        print("⚠️ Nenhuma temperatura possui os três danos nos dois métodos. Histograma ignorado.")
        return None

    if len(temps_validas) > N_TEMPS_HIST:
        rng = np.random.default_rng(SEED_TEMPS)
        temps_validas = sorted(rng.choice(temps_validas, N_TEMPS_HIST, replace=False))

    colors = [CORES_DANO[0], CORES_DANO[1], CORES_DANO[2]]
    x = np.arange(len(temps_validas))

    bar_w = 0.12
    gap = 0.10
    rf_offsets = np.array([0, 1, 2]) * bar_w
    pk_offsets = (3 * bar_w + gap) + np.array([0, 1, 2]) * bar_w
    x_center = x + ((rf_offsets.mean() + pk_offsets.mean()) / 2)

    def medias(df, metric):
        out = {d: [] for d in danos}
        for d in danos:
            for T in temps_validas:
                mask = (df["falha"] == d) & np.isclose(df["temperatura_c"], T)
                out[d].append(df.loc[mask, metric].mean() if np.any(mask) else np.nan)
        return out

    rmsd_rf = medias(df_rf, "RMSD")
    rmsd_pk = medias(df_park, "RMSD")
    ccdm_rf = medias(df_rf, "CCDM")
    ccdm_pk = medias(df_park, "CCDM")

    fig, axes = plt.subplots(1, 2, figsize=(24, 8.5), dpi=300)

    for ax in axes:
        estilo_eixos(ax)

    ax = axes[0]
    for i, d in enumerate(danos):
        ax.bar(x + rf_offsets[i], rmsd_rf[d], width=bar_w, color=colors[i], alpha=0.55,
               edgecolor="black", linewidth=0.9, label=f"RF — dano {d}")
        ax.bar(x + pk_offsets[i], rmsd_pk[d], width=bar_w, color=colors[i], alpha=1.0,
               edgecolor="black", linewidth=0.9, label=f"Park — dano {d}")

    ax.set_ylabel("RMSD", fontsize=28)
    ax.set_xlabel("Temperatura (°C)", fontsize=28, labelpad=12)
    ax.set_xticks(x_center)
    ax.set_xticklabels([f"{int(t)}" if float(t).is_integer() else f"{t:.1f}" for t in temps_validas])
    ax.set_title("(a) RMSD", fontsize=28, pad=16)

    ax = axes[1]
    for i, d in enumerate(danos):
        ax.bar(x + rf_offsets[i], ccdm_rf[d], width=bar_w, color=colors[i], alpha=0.55,
               edgecolor="black", linewidth=0.9, label=f"RF — dano {d}")
        ax.bar(x + pk_offsets[i], ccdm_pk[d], width=bar_w, color=colors[i], alpha=1.0,
               edgecolor="black", linewidth=0.9, label=f"Park — dano {d}")

    ax.set_ylabel("CCDM", fontsize=28)
    ax.set_xlabel("Temperatura (°C)", fontsize=28, labelpad=12)
    ax.set_xticks(x_center)
    ax.set_xticklabels([f"{int(t)}" if float(t).is_integer() else f"{t:.1f}" for t in temps_validas])
    ax.set_title("(b) CCDM", fontsize=28, pad=16)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=3, frameon=True,
               fontsize=17, bbox_to_anchor=(0.5, -0.08))

    fig.suptitle(
        f"RF ponto a ponto vs Park — {fmin_khz:.0f}–{fmax_khz:.0f} kHz — referência {REF_TEMP}°C",
        fontsize=30,
        y=1.02,
    )

    fig.tight_layout(rect=[0, 0.08, 1, 0.95])

    nome = f"hist_rf_park_{fmin_khz:.0f}_{fmax_khz:.0f}kHz"
    salvar_fig(fig, output_dir, nome)
    plt.close(fig)

    return fig


def plot_ranking_faixas(df_ranking, top_n=15):
    top = df_ranking.head(top_n).copy()
    top["faixa"] = top["faixa_label"] + "\n" + top["metodo"].map(NOME_METODO).fillna(top["metodo"])

    fig, ax = plt.subplots(figsize=(18, 8), dpi=300)

    ax.bar(top["faixa"], top["score"], edgecolor="black", linewidth=0.9)
    ax.set_ylabel("Score combinado menor = melhor", fontsize=26)
    ax.set_xlabel("Faixa e método", fontsize=26)
    ax.set_title("Ranking das melhores faixas de frequência", fontsize=28, pad=16)
    ax.tick_params(axis="x", rotation=45, labelsize=18)
    ax.tick_params(axis="y", labelsize=22)
    estilo_eixos(ax)

    fig.tight_layout()
    salvar_fig(fig, PASTA_GRAF, "ranking_melhores_faixas")
    plt.close(fig)

    return fig


def plot_metrica_por_faixa_por_metodo(df, metrica="RMSD"):
    col_media = f"{metrica}_medio"
    col_std = f"{metrica}_std"

    metodos = ["RF_ponto_a_ponto", "Park"]
    danos = sorted(df["falha"].unique())

    fig, axes = plt.subplots(1, 2, figsize=(24, 8), dpi=300, sharey=True)

    for ax, metodo in zip(axes, metodos):
        sub_met = df[df["metodo"] == metodo]

        for dano in danos:
            sub = sub_met[sub_met["falha"] == dano].sort_values("faixa_centro_khz")
            y, yerr = erro_valido(sub[col_media], sub[col_std])

            ax.errorbar(
                sub["faixa_centro_khz"],
                y,
                yerr=yerr,
                marker="o",
                linewidth=2.8,
                markersize=8,
                capsize=4,
                color=CORES_DANO.get(int(dano), None),
                label=f"Dano {int(dano)}",
            )

        ax.set_title(NOME_METODO.get(metodo, metodo), fontsize=28, pad=14)
        ax.set_xlabel("Centro da faixa (kHz)", fontsize=26)
        estilo_eixos(ax)

    axes[0].set_ylabel(metrica, fontsize=28)
    fig.suptitle(f"{metrica} médio em faixas de 20 kHz", fontsize=32, y=1.03)
    axes[1].legend(frameon=True, fontsize=18, loc="best")

    fig.tight_layout()
    salvar_fig(fig, PASTA_GRAF, f"{metrica.lower()}_medio_20khz_por_metodo")
    plt.close(fig)

    return fig


def plot_metrica_rf_vs_park_por_dano(df, metrica="RMSD"):
    col_media = f"{metrica}_medio"
    col_std = f"{metrica}_std"

    danos = sorted(df["falha"].unique())
    metodos = ["RF_ponto_a_ponto", "Park"]

    fig, axes = plt.subplots(1, len(danos), figsize=(24, 7.5), dpi=300, sharey=True)

    if len(danos) == 1:
        axes = [axes]

    for ax, dano in zip(axes, danos):
        sub_d = df[df["falha"] == dano]

        for metodo in metodos:
            sub = sub_d[sub_d["metodo"] == metodo].sort_values("faixa_centro_khz")
            y, yerr = erro_valido(sub[col_media], sub[col_std])

            ax.errorbar(
                sub["faixa_centro_khz"],
                y,
                yerr=yerr,
                marker=MARCADORES_METODO.get(metodo, "o"),
                linestyle=ESTILO_METODO.get(metodo, "-"),
                linewidth=3.0,
                markersize=8,
                capsize=4,
                label=NOME_METODO.get(metodo, metodo),
            )

        ax.set_title(f"Dano {int(dano)}", fontsize=28, pad=14)
        ax.set_xlabel("Centro da faixa (kHz)", fontsize=25)
        estilo_eixos(ax)

    axes[0].set_ylabel(metrica, fontsize=28)
    axes[-1].legend(frameon=True, fontsize=18, loc="best")

    fig.suptitle(f"{metrica}: RF ponto a ponto vs Park em faixas de 20 kHz", fontsize=32, y=1.04)
    fig.tight_layout()
    salvar_fig(fig, PASTA_GRAF, f"{metrica.lower()}_rf_vs_park_por_dano_20khz")
    plt.close(fig)

    return fig


def plot_separacao_entre_danos(df_ranking):
    metodos = ["RF_ponto_a_ponto", "Park"]

    for metrica in ["RMSD", "CCDM"]:
        fig, axes = plt.subplots(1, 2, figsize=(22, 7), dpi=300, sharey=True)

        for ax, metodo in zip(axes, metodos):
            sub = df_ranking[df_ranking["metodo"] == metodo].sort_values("faixa_centro_khz")

            ax.plot(
                sub["faixa_centro_khz"],
                sub[f"sep_{metrica}_D1_D0"],
                marker="o",
                linewidth=3,
                label="D1 - D0",
            )
            ax.plot(
                sub["faixa_centro_khz"],
                sub[f"sep_{metrica}_D2_D1"],
                marker="s",
                linewidth=3,
                label="D2 - D1",
            )
            ax.axhline(0, linestyle="--", linewidth=1.5)
            ax.set_title(NOME_METODO.get(metodo, metodo), fontsize=28, pad=14)
            ax.set_xlabel("Centro da faixa (kHz)", fontsize=25)
            estilo_eixos(ax)

        axes[0].set_ylabel(f"Separação em {metrica}", fontsize=28)
        axes[-1].legend(frameon=True, fontsize=18, loc="best")
        fig.suptitle(f"Separação entre danos por faixa — {metrica}", fontsize=32, y=1.04)
        fig.tight_layout()
        salvar_fig(fig, PASTA_GRAF, f"separacao_danos_{metrica.lower()}_20khz")
        plt.close(fig)


# ============================================================
# 12) HEATMAPS
# ============================================================

def heatmap_from_pivot(pivot, titulo, cbar_label, nome_base, pasta=PASTA_HEAT):
    """Cria heatmap com matplotlib imshow, sem seaborn."""
    data = pivot.to_numpy(dtype=float)

    fig, ax = plt.subplots(figsize=(max(12, 1.0 * len(pivot.columns)), 6.8), dpi=300)

    im = ax.imshow(data, aspect="auto")

    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_yticks(np.arange(len(pivot.index)))

    ax.set_xticklabels(pivot.columns, rotation=45, ha="right", fontsize=17)
    ax.set_yticklabels(pivot.index, fontsize=20)

    ax.set_title(titulo, fontsize=28, pad=18)
    ax.set_xlabel("Faixa de frequência", fontsize=25)
    ax.set_ylabel("Dano", fontsize=25)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(cbar_label, fontsize=22)
    cbar.ax.tick_params(labelsize=16)

    # Escreve os valores dentro das células
    finite_data = data[np.isfinite(data)]
    if len(finite_data) > 0:
        meio = 0.5 * (np.nanmin(finite_data) + np.nanmax(finite_data))
    else:
        meio = 0.0

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            val = data[i, j]
            if np.isfinite(val):
                txt_color = "white" if val > meio else "black"
                ax.text(j, i, f"{val:.3g}", ha="center", va="center", fontsize=13, color=txt_color)

    fig.tight_layout()
    salvar_fig(fig, pasta, nome_base)
    plt.close(fig)

    return fig


def gerar_heatmaps_metricas(df_resumo):
    """Gera heatmaps RMSD e CCDM para cada método."""
    df = df_resumo.copy()
    df["faixa_label"] = (
        df["faixa_min_khz"].round(0).astype(int).astype(str)
        + "-"
        + df["faixa_max_khz"].round(0).astype(int).astype(str)
        + " kHz"
    )
    df["dano_label"] = "Dano " + df["falha"].astype(int).astype(str)

    for metodo in ["RF_ponto_a_ponto", "Park"]:
        sub = df[df["metodo"] == metodo].copy()
        metodo_nome = NOME_METODO.get(metodo, metodo)

        for metrica in ["RMSD", "CCDM"]:
            pivot = sub.pivot_table(
                index="dano_label",
                columns="faixa_label",
                values=f"{metrica}_medio",
                aggfunc="mean",
            )

            # Garante ordem Dano 0, 1, 2
            ordem_index = [f"Dano {d}" for d in sorted(df["falha"].unique())]
            pivot = pivot.reindex(ordem_index)

            # Ordena colunas pela frequência inicial
            labels_ordenadas = (
                sub[["faixa_label", "faixa_min_khz"]]
                .drop_duplicates()
                .sort_values("faixa_min_khz")["faixa_label"]
                .tolist()
            )
            pivot = pivot[labels_ordenadas]

            heatmap_from_pivot(
                pivot,
                titulo=f"Heatmap de {metrica} médio — {metodo_nome}",
                cbar_label=f"{metrica} médio",
                nome_base=f"heatmap_{metrica.lower()}_{metodo}_20khz",
            )


def gerar_heatmap_score(df_ranking):
    """Heatmap do score combinado por método e faixa."""
    df = df_ranking.copy()
    df["metodo_nome"] = df["metodo"].map(NOME_METODO).fillna(df["metodo"])
    df["faixa_label"] = df["faixa_label"].astype(str)

    pivot = df.pivot_table(
        index="metodo_nome",
        columns="faixa_label",
        values="score",
        aggfunc="mean",
    )

    labels_ordenadas = (
        df[["faixa_label", "faixa_min_khz"]]
        .drop_duplicates()
        .sort_values("faixa_min_khz")["faixa_label"]
        .tolist()
    )
    pivot = pivot[labels_ordenadas]

    heatmap_from_pivot(
        pivot,
        titulo="Heatmap do score combinado por faixa",
        cbar_label="Score menor = melhor",
        nome_base="heatmap_score_combinado_20khz",
    )


def gerar_heatmaps_separacao(df_ranking):
    """Heatmaps das separações D1-D0 e D2-D1 para RMSD e CCDM."""
    df = df_ranking.copy()
    df["metodo_nome"] = df["metodo"].map(NOME_METODO).fillna(df["metodo"])

    labels_ordenadas = (
        df[["faixa_label", "faixa_min_khz"]]
        .drop_duplicates()
        .sort_values("faixa_min_khz")["faixa_label"]
        .tolist()
    )

    for metrica in ["RMSD", "CCDM"]:
        for sep_nome, sep_col in [
            ("D1 - D0", f"sep_{metrica}_D1_D0"),
            ("D2 - D1", f"sep_{metrica}_D2_D1"),
        ]:
            pivot = df.pivot_table(
                index="metodo_nome",
                columns="faixa_label",
                values=sep_col,
                aggfunc="mean",
            )
            pivot = pivot[labels_ordenadas]

            heatmap_from_pivot(
                pivot,
                titulo=f"Heatmap da separação {sep_nome} — {metrica}",
                cbar_label=f"Separação {sep_nome}",
                nome_base=f"heatmap_separacao_{metrica.lower()}_{sep_nome.replace(' ', '').replace('-', '_')}_20khz",
            )


# ============================================================
# 13) VARREDURA COMPLETA
# ============================================================

def varrer_faixas_20khz():
    print("🔹 Carregando base...")
    df_base = pd.read_pickle(ARQ_BASE)

    required = ["temperatura_c", "falha"]
    missing = [c for c in required if c not in df_base.columns]
    if missing:
        raise ValueError(f"A base não tem as colunas obrigatórias: {missing}")

    if USAR_FAIXA_TOTAL_DA_BASE:
        fmin_global, fmax_global = definir_limites_frequencia(df_base)
    else:
        fmin_global, fmax_global = float(FREQ_MIN_GLOBAL_KHZ), float(FREQ_MAX_GLOBAL_KHZ)

    print(f"📌 Faixa global usada: {fmin_global:.2f} até {fmax_global:.2f} kHz")
    print(f"📌 Largura da janela: {LARGURA_FAIXA_KHZ} kHz")
    print(f"📌 Passo entre janelas: {PASSO_FAIXA_KHZ} kHz")

    faixas = gerar_faixas(
        fmin_global,
        fmax_global,
        LARGURA_FAIXA_KHZ,
        PASSO_FAIXA_KHZ,
        incluir_final=INCLUIR_FAIXA_FINAL_INCOMPLETA,
    )

    print(f"🔎 Total de faixas a testar: {len(faixas)}")

    todos_resumos = []
    metricas_rf_por_faixa = []
    metricas_park_por_faixa = []
    erros = []

    for fmin, fmax in faixas:
        try:
            df_rf_met, df_park_met, df_resumo = executar_uma_faixa(df_base, fmin, fmax)

            todos_resumos.append(df_resumo)

            df_rf_save = df_rf_met.copy()
            df_rf_save["faixa_min_khz"] = fmin
            df_rf_save["faixa_max_khz"] = fmax
            df_rf_save["metodo"] = "RF_ponto_a_ponto"
            metricas_rf_por_faixa.append(df_rf_save)

            df_park_save = df_park_met.copy()
            df_park_save["faixa_min_khz"] = fmin
            df_park_save["faixa_max_khz"] = fmax
            df_park_save["metodo"] = "Park"
            metricas_park_por_faixa.append(df_park_save)

            if GERAR_HISTOGRAMAS_TODAS_AS_FAIXAS:
                histograma_rf_park_temperaturas_validas_2painel(
                    df_rf_met,
                    df_park_met,
                    fmin,
                    fmax,
                    output_dir=PASTA_HIST,
                )

        except Exception as e:
            print(f"⚠️ Erro na faixa {fmin:.1f}-{fmax:.1f} kHz: {e}")
            erros.append({"faixa_min_khz": fmin, "faixa_max_khz": fmax, "erro": str(e)})

    if len(todos_resumos) == 0:
        raise RuntimeError("Nenhuma faixa foi processada com sucesso.")

    df_resumo_total = pd.concat(todos_resumos, ignore_index=True)
    df_ranking = montar_tabela_ranking(df_resumo_total)

    df_metricas_todas = pd.concat(metricas_rf_por_faixa + metricas_park_por_faixa, ignore_index=True)

    path_resumo = os.path.join(PASTA_SAIDA, "resumo_metricas_todas_faixas_20khz.csv")
    path_ranking = os.path.join(PASTA_SAIDA, "ranking_melhores_faixas_20khz.csv")
    path_metricas = os.path.join(PASTA_SAIDA, "metricas_amostra_a_amostra_todas_faixas_20khz.csv")
    path_erros = os.path.join(PASTA_SAIDA, "erros_varredura_20khz.csv")

    df_resumo_total.to_csv(path_resumo, index=False)
    df_ranking.to_csv(path_ranking, index=False)
    df_metricas_todas.to_csv(path_metricas, index=False)

    if len(erros) > 0:
        pd.DataFrame(erros).to_csv(path_erros, index=False)

    print("\n✅ Varredura finalizada.")
    print(f"Resumo salvo em: {path_resumo}")
    print(f"Ranking salvo em: {path_ranking}")
    print(f"Métricas amostra a amostra salvas em: {path_metricas}")

    if len(erros) > 0:
        print(f"⚠️ Algumas faixas deram erro. Veja: {path_erros}")

    print("\n🏆 TOP 10 melhores faixas pelo critério combinado:")
    print(df_ranking.head(10).to_string(index=False))

    return df_base, df_resumo_total, df_ranking, df_metricas_todas


# ============================================================
# 14) RODAR TUDO
# ============================================================

if __name__ == "__main__":

    # 1) Calcula RF e Park em todas as faixas de 20 kHz
    df_base, df_resumo_total, df_ranking, df_metricas_todas = varrer_faixas_20khz()

    # 2) Gráficos de ranking e métricas por faixa
    plot_ranking_faixas(df_ranking, top_n=15)

    plot_metrica_por_faixa_por_metodo(df_resumo_total, metrica="RMSD")
    plot_metrica_por_faixa_por_metodo(df_resumo_total, metrica="CCDM")

    plot_metrica_rf_vs_park_por_dano(df_resumo_total, metrica="RMSD")
    plot_metrica_rf_vs_park_por_dano(df_resumo_total, metrica="CCDM")

    plot_separacao_entre_danos(df_ranking)

    # 3) Heatmaps principais
    gerar_heatmaps_metricas(df_resumo_total)
    gerar_heatmap_score(df_ranking)
    gerar_heatmaps_separacao(df_ranking)

    # 4) Histograma extra da melhor faixa geral, se quiser destacar no slide
    if GERAR_HISTOGRAMA_MELHOR_FAIXA:
        melhor = df_ranking.iloc[0]
        melhor_fmin = float(melhor["faixa_min_khz"])
        melhor_fmax = float(melhor["faixa_max_khz"])

        print("\n🏆 Melhor faixa encontrada:")
        print(melhor.to_string())

        df_rf_best, df_park_best, _ = executar_uma_faixa(df_base, melhor_fmin, melhor_fmax)

        histograma_rf_park_temperaturas_validas_2painel(
            df_rf_best,
            df_park_best,
            melhor_fmin,
            melhor_fmax,
            output_dir=PASTA_SAIDA,
        )

        df_rf_best.to_csv(
            os.path.join(PASTA_SAIDA, f"metricas_RF_melhor_faixa_{melhor_fmin:.0f}_{melhor_fmax:.0f}kHz.csv"),
            index=False,
        )

        df_park_best.to_csv(
            os.path.join(PASTA_SAIDA, f"metricas_Park_melhor_faixa_{melhor_fmin:.0f}_{melhor_fmax:.0f}kHz.csv"),
            index=False,
        )

    print("\n✅ Tudo finalizado.")
    print(f"📁 Resultados em: {PASTA_SAIDA}")


In [ ]:
# ============================================================
# CÓDIGO ADICIONAL — PLOTAR RMSD E CCDM DE TODAS AS FAIXAS
# ============================================================
# Use este código DEPOIS de rodar o código principal da varredura.
# Ele lê o arquivo:
#   resultados_varredura_rf_park/resumo_metricas_todas_faixas.csv
#
# E gera gráficos grandes para usar em slides/artigo:
#   1) RMSD médio por faixa — RF e Park separados
#   2) CCDM médio por faixa — RF e Park separados
#   3) RMSD médio comparando RF vs Park para cada dano
#   4) CCDM médio comparando RF vs Park para cada dano
#   5) Separação entre danos por faixa
#   6) Heatmaps de RMSD e CCDM por método/dano/faixa
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1) CONFIGURAÇÕES
# ------------------------------------------------------------

PASTA_SAIDA = "resultados_varredura_rf_park"
ARQ_RESUMO = os.path.join(PASTA_SAIDA, "resumo_metricas_todas_faixas.csv")
PASTA_GRAFICOS = os.path.join(PASTA_SAIDA, "graficos_todas_as_faixas")
os.makedirs(PASTA_GRAFICOS, exist_ok=True)

# Escolha True para salvar também em PDF
SALVAR_PDF = True

# Visual maior, bom para apresentação/artigo
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 22,
    "axes.labelsize": 26,
    "axes.titlesize": 28,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 18,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# Nomes bonitos para legenda
NOME_METODO = {
    "RF_ponto_a_ponto": "RF ponto a ponto",
    "Park": "Park"
}

# Cores por dano
CORES_DANO = {
    0: "tab:blue",
    1: "tab:orange",
    2: "tab:red"
}

# Estilos por método
ESTILO_METODO = {
    "RF_ponto_a_ponto": "-",
    "Park": "--"
}

MARCADORES_METODO = {
    "RF_ponto_a_ponto": "o",
    "Park": "s"
}


# ------------------------------------------------------------
# 2) FUNÇÕES AUXILIARES
# ------------------------------------------------------------

def preparar_resumo(path=ARQ_RESUMO):
    """Carrega o CSV da varredura e cria colunas úteis para plot."""
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Não encontrei o arquivo {path}.\n"
            "Rode primeiro o código principal da varredura para gerar o CSV."
        )

    df = pd.read_csv(path)

    cols_obrigatorias = [
        "faixa_min_khz", "faixa_max_khz", "metodo", "falha",
        "RMSD_medio", "RMSD_std", "CCDM_medio", "CCDM_std"
    ]
    faltando = [c for c in cols_obrigatorias if c not in df.columns]
    if faltando:
        raise ValueError(f"O CSV não tem as colunas obrigatórias: {faltando}")

    df["faixa_centro_khz"] = 0.5 * (df["faixa_min_khz"] + df["faixa_max_khz"])
    df["faixa_label"] = (
        df["faixa_min_khz"].astype(int).astype(str)
        + "–"
        + df["faixa_max_khz"].astype(int).astype(str)
        + " kHz"
    )

    df = df.sort_values(["faixa_min_khz", "faixa_max_khz", "metodo", "falha"]).reset_index(drop=True)
    return df


def salvar_fig(fig, nome_base):
    """Salva figura em PNG e, opcionalmente, PDF."""
    png_path = os.path.join(PASTA_GRAFICOS, nome_base + ".png")
    fig.savefig(png_path, dpi=600, bbox_inches="tight")

    if SALVAR_PDF:
        pdf_path = os.path.join(PASTA_GRAFICOS, nome_base + ".pdf")
        fig.savefig(pdf_path, bbox_inches="tight")
        print(f"✅ Salvo:\n{png_path}\n{pdf_path}")
    else:
        print(f"✅ Salvo:\n{png_path}")


def estilo_eixos(ax):
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=20)


def erro_valido(y, yerr):
    """Evita erro NaN no errorbar."""
    y = np.asarray(y, dtype=float)
    yerr = np.asarray(yerr, dtype=float)
    yerr = np.where(np.isfinite(yerr), yerr, 0.0)
    return y, yerr


# ------------------------------------------------------------
# 3) PLOT 1 — RMSD OU CCDM POR FAIXA, SEPARADO POR MÉTODO
# ------------------------------------------------------------

def plot_metrica_por_faixa_por_metodo(df, metrica="RMSD"):
    """
    Gera 1 figura com 2 painéis:
    - esquerda: RF ponto a ponto
    - direita: Park

    Cada linha representa um dano.
    """
    col_media = f"{metrica}_medio"
    col_std = f"{metrica}_std"

    metodos = ["RF_ponto_a_ponto", "Park"]
    danos = sorted(df["falha"].unique())

    fig, axes = plt.subplots(1, 2, figsize=(24, 8), dpi=300, sharey=True)

    for ax, metodo in zip(axes, metodos):
        sub_met = df[df["metodo"] == metodo]

        for dano in danos:
            sub = sub_met[sub_met["falha"] == dano].sort_values("faixa_centro_khz")

            y, yerr = erro_valido(sub[col_media], sub[col_std])

            ax.errorbar(
                sub["faixa_centro_khz"],
                y,
                yerr=yerr,
                marker="o",
                linewidth=2.8,
                markersize=8,
                capsize=4,
                color=CORES_DANO.get(int(dano), None),
                label=f"Dano {int(dano)}"
            )

        ax.set_title(NOME_METODO.get(metodo, metodo), fontsize=28, pad=14)
        ax.set_xlabel("Centro da faixa de frequência (kHz)", fontsize=26)
        estilo_eixos(ax)

    axes[0].set_ylabel(metrica, fontsize=28)

    fig.suptitle(f"{metrica} médio em todas as faixas", fontsize=32, y=1.03)
    axes[1].legend(frameon=True, fontsize=18, loc="best")

    fig.tight_layout()
    salvar_fig(fig, f"{metrica.lower()}_medio_todas_faixas_por_metodo")
    plt.show()

    return fig


# ------------------------------------------------------------
# 4) PLOT 2 — COMPARAÇÃO RF vs PARK PARA CADA DANO
# ------------------------------------------------------------

def plot_metrica_rf_vs_park_por_dano(df, metrica="RMSD"):
    """
    Gera 1 figura com 3 painéis:
    - dano 0
    - dano 1
    - dano 2

    Em cada painel aparecem RF ponto a ponto e Park.
    """
    col_media = f"{metrica}_medio"
    col_std = f"{metrica}_std"

    danos = sorted(df["falha"].unique())
    metodos = ["RF_ponto_a_ponto", "Park"]

    fig, axes = plt.subplots(1, len(danos), figsize=(8 * len(danos), 7.5), dpi=300, sharey=True)

    if len(danos) == 1:
        axes = [axes]

    for ax, dano in zip(axes, danos):
        sub_dano = df[df["falha"] == dano]

        for metodo in metodos:
            sub = sub_dano[sub_dano["metodo"] == metodo].sort_values("faixa_centro_khz")
            y, yerr = erro_valido(sub[col_media], sub[col_std])

            ax.errorbar(
                sub["faixa_centro_khz"],
                y,
                yerr=yerr,
                linestyle=ESTILO_METODO.get(metodo, "-"),
                marker=MARCADORES_METODO.get(metodo, "o"),
                linewidth=2.8,
                markersize=8,
                capsize=4,
                label=NOME_METODO.get(metodo, metodo)
            )

        ax.set_title(f"Dano {int(dano)}", fontsize=28, pad=14)
        ax.set_xlabel("Centro da faixa (kHz)", fontsize=26)
        estilo_eixos(ax)

    axes[0].set_ylabel(metrica, fontsize=28)
    axes[-1].legend(frameon=True, fontsize=18, loc="best")

    fig.suptitle(f"Comparação RF vs Park — {metrica} em todas as faixas", fontsize=32, y=1.03)
    fig.tight_layout()

    salvar_fig(fig, f"{metrica.lower()}_rf_vs_park_por_dano_todas_faixas")
    plt.show()

    return fig


# ------------------------------------------------------------
# 5) PLOT 3 — SEPARAÇÃO ENTRE DANOS POR FAIXA
# ------------------------------------------------------------

def calcular_separacoes(df):
    """
    Calcula separações médias entre danos:
    - D1 - D0
    - D2 - D1
    - D2 - D0

    Isso ajuda a ver se a faixa preserva assinatura de dano.
    """
    rows = []

    for (fmin, fmax, metodo), g in df.groupby(["faixa_min_khz", "faixa_max_khz", "metodo"]):
        gd = g.set_index("falha")

        if not all(d in gd.index for d in [0, 1, 2]):
            continue

        for metrica in ["RMSD", "CCDM"]:
            v0 = gd.loc[0, f"{metrica}_medio"]
            v1 = gd.loc[1, f"{metrica}_medio"]
            v2 = gd.loc[2, f"{metrica}_medio"]

            rows.append({
                "faixa_min_khz": fmin,
                "faixa_max_khz": fmax,
                "faixa_centro_khz": 0.5 * (fmin + fmax),
                "metodo": metodo,
                "metrica": metrica,
                "sep_D1_D0": v1 - v0,
                "sep_D2_D1": v2 - v1,
                "sep_D2_D0": v2 - v0,
                "inversao_D1_D0": int((v1 - v0) <= 0),
                "inversao_D2_D1": int((v2 - v1) <= 0),
            })

    return pd.DataFrame(rows)


def plot_separacao_danos(df, metrica="RMSD"):
    """
    Plota separação entre danos em todas as faixas.

    Interpretação:
    - Valores positivos são bons.
    - Se D1-D0 ou D2-D1 ficar negativo, há inversão da ordem esperada.
    """
    sep = calcular_separacoes(df)
    sep = sep[sep["metrica"] == metrica].copy()

    metodos = ["RF_ponto_a_ponto", "Park"]
    variaveis = ["sep_D1_D0", "sep_D2_D1", "sep_D2_D0"]
    labels = {
        "sep_D1_D0": "Dano 1 − Dano 0",
        "sep_D2_D1": "Dano 2 − Dano 1",
        "sep_D2_D0": "Dano 2 − Dano 0"
    }

    fig, axes = plt.subplots(1, 2, figsize=(24, 8), dpi=300, sharey=True)

    for ax, metodo in zip(axes, metodos):
        sub_m = sep[sep["metodo"] == metodo].sort_values("faixa_centro_khz")

        for var in variaveis:
            ax.plot(
                sub_m["faixa_centro_khz"],
                sub_m[var],
                marker="o",
                linewidth=2.8,
                markersize=8,
                label=labels[var]
            )

        ax.axhline(0, color="black", linewidth=1.2)
        ax.set_title(NOME_METODO.get(metodo, metodo), fontsize=28, pad=14)
        ax.set_xlabel("Centro da faixa (kHz)", fontsize=26)
        estilo_eixos(ax)

    axes[0].set_ylabel(f"Separação em {metrica}", fontsize=28)
    axes[1].legend(frameon=True, fontsize=18, loc="best")

    fig.suptitle(f"Separação entre danos por faixa — {metrica}", fontsize=32, y=1.03)
    fig.tight_layout()

    salvar_fig(fig, f"separacao_danos_{metrica.lower()}_todas_faixas")
    plt.show()

    return fig


# ------------------------------------------------------------
# 6) PLOT 4 — HEATMAP DE MÉTRICA POR FAIXA E DANO
# ------------------------------------------------------------

def plot_heatmap_metrica(df, metrica="RMSD", metodo="RF_ponto_a_ponto"):
    """
    Heatmap simples:
    - eixo x: faixa de frequência
    - eixo y: dano
    - cor: RMSD ou CCDM médio
    """
    col_media = f"{metrica}_medio"

    sub = df[df["metodo"] == metodo].copy()
    sub = sub.sort_values(["faixa_min_khz", "falha"])

    pivot = sub.pivot_table(
        index="falha",
        columns="faixa_label",
        values=col_media,
        aggfunc="mean"
    )

    # Reordena colunas pela frequência mínima
    ordem = (
        sub[["faixa_label", "faixa_min_khz"]]
        .drop_duplicates()
        .sort_values("faixa_min_khz")["faixa_label"]
        .tolist()
    )
    pivot = pivot[ordem]

    fig, ax = plt.subplots(figsize=(max(14, 0.9 * len(ordem)), 6.5), dpi=300)

    im = ax.imshow(pivot.values, aspect="auto")

    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels([f"Dano {int(d)}" for d in pivot.index])

    ax.set_xlabel("Faixa de frequência", fontsize=26)
    ax.set_ylabel("Estado estrutural", fontsize=26)
    ax.set_title(f"{metrica} médio — {NOME_METODO.get(metodo, metodo)}", fontsize=28, pad=16)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(metrica, fontsize=24)
    cbar.ax.tick_params(labelsize=18)

    # Escreve valores dentro das células
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.values[i, j]
            if np.isfinite(val):
                ax.text(j, i, f"{val:.3g}", ha="center", va="center", fontsize=13)

    fig.tight_layout()

    nome_metodo = metodo.lower().replace("_", "-")
    salvar_fig(fig, f"heatmap_{metrica.lower()}_{nome_metodo}_todas_faixas")
    plt.show()

    return fig


# ------------------------------------------------------------
# 7) RODAR TODOS OS GRÁFICOS
# ------------------------------------------------------------

def plotar_tudo_todas_as_faixas():
    df = preparar_resumo(ARQ_RESUMO)

    # 1) RMSD e CCDM por faixa, com RF e Park separados
    plot_metrica_por_faixa_por_metodo(df, metrica="RMSD")
    plot_metrica_por_faixa_por_metodo(df, metrica="CCDM")

    # 2) RMSD e CCDM comparando RF vs Park dentro de cada dano
    plot_metrica_rf_vs_park_por_dano(df, metrica="RMSD")
    plot_metrica_rf_vs_park_por_dano(df, metrica="CCDM")

    # 3) Separação entre danos
    plot_separacao_danos(df, metrica="RMSD")
    plot_separacao_danos(df, metrica="CCDM")

    # 4) Heatmaps
    for metodo in ["RF_ponto_a_ponto", "Park"]:
        plot_heatmap_metrica(df, metrica="RMSD", metodo=metodo)
        plot_heatmap_metrica(df, metrica="CCDM", metodo=metodo)

    print("\n✅ Todos os gráficos de todas as faixas foram gerados.")
    print(f"📁 Pasta de saída: {PASTA_GRAFICOS}")

    return df


# ------------------------------------------------------------
# 8) EXECUÇÃO
# ------------------------------------------------------------

if __name__ == "__main__":
    df_todas_as_faixas = plotar_tudo_todas_as_faixas()
